In [ ]:
import os
import sys
import numpy as np
import torch
import scipy.signal as signal
# Nos aseguramos de tener acceso a tus algoritmos
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.dsp_rf.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico
from NoisyUAV.modelos.cvcnn import ComplexConv1DNet
import torch.nn.functional as F
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import SNR_estimation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import matplotlib.gridspec as gridspec
from scipy import ndimage

from NoisyUAV.modelo_xin_v1.xin_cvcnn import XinCVCNN
from NoisyUAV.modelo_xin_v1.xin_dataset import (
    iq_to_xin_tensor, NFFT, HOP_LENGTH, SPEC_H, SPEC_W, DB_CLIP
)

In [ ]:
# ── 1. CONFIGURACIÓN DE RUTAS Y PARÁMETROS ───────────────────────────────────
# Ruta al archivo específico que quieres probar
archivo_rfuav = r"C:\TFM_data\RFUAV\JUMPER_20TProV2\JUMPER_TProV2\pack1_3-4s.iq"
# Parámetros de frecuencia
FS_ORIGINAL = 100e6  # 100 MHz (RFUAV)
FS_TARGET   = 14e6   # 14 MHz (Tu modelo)
# Cuánto tiempo queremos leer (ms)
MS_A_LEER = 1000
MUESTRAS_COMPLEJAS = int((MS_A_LEER / 1000) * FS_ORIGINAL)
TOTAL_FLOATS = MUESTRAS_COMPLEJAS * 2

In [ ]:
# 2. CARGA Y DESENTRELAZADO (I, Q)
if not os.path.exists(archivo_rfuav):
    raise FileNotFoundError(f"No se encuentra el archivo en la nueva ruta: {archivo_rfuav}")
datos_raw = np.fromfile(archivo_rfuav, dtype=np.float32, count=TOTAL_FLOATS)
# Desentrelazar (IQIQIQ... -> I y Q separados)
I_orig = datos_raw[0::2]
Q_orig = datos_raw[1::2]
print(f"✅ Cargado. Arrays originales: I={I_orig.shape}, Q={Q_orig.shape}")
# Añadir un AGC básico temporal
max_val = max(np.max(np.abs(I_orig)), np.max(np.abs(Q_orig)))
I_orig = I_orig / max_val
Q_orig = Q_orig / max_val
S = I_orig + 1j * Q_orig

In [ ]:
# Definimos la SNR que queremos probar (ejemplo -10 dB)
SNR_A_PROBAR = -2 
bajar_snr = 1  
# Convertimos a complejo para aplicar el ruido IQ correctamente
if bajar_snr:
    # Llamamos a tu función modificada
    S = SNR_estimation.add_awgn_noise(S, SNR_A_PROBAR)
# Extraemos de nuevo I y Q para seguir con tu flujo
I_orig = np.real(S).astype(np.float32)
Q_orig = np.imag(S).astype(np.float32)
print(f"✅ Señal inyectada con ruido AWGN Complejo. Nueva SNR: {SNR_A_PROBAR} dB")

In [ ]:
# 3. DOWNSAMPLING (de 100 MHz a 14 MHz)
print(f"⏳ Aplicando filtro decimador polifásico (100MHz -> 14MHz)...")
# Para ir de 100 a 14 multiplicamos por 14 y dividimos por 100 (up=7, down=50)
I_resampled = signal.resample_poly(I_orig, up=7, down=50)
Q_resampled = signal.resample_poly(Q_orig, up=7, down=50)
# Convertir al formato tensor float de Pytorch [2, N]
iq_tensor = torch.tensor(np.array([I_resampled, Q_resampled]), dtype=torch.float32)
print(f"✅ Downsampling Completado. Tensor Final: {iq_tensor.shape}")

In [ ]:
# Variables Físicas de la Tesis
FS = 14e6
NPERSEG = 2048
Z_THRESH = 4.0     
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 0.75
MIN_Z_ABS = 4.0
BG_MULT = 4
MAX_BINS_FRAC = 1
SMOOTH_MS = 0.1
ADAPTIVE_WINDOW_MS = 10 

# FS = 14e6
# NPERSEG = 2048
# Z_THRESH = 1.0       # <--- SUPER RELAJADO PARA CAZAR EL FONDO DEL RUIDO
# MIN_BURST_MS = 0.25  # <--- Más corto permitido
# MERGE_GAP_MS = 0.5
# MIN_Z_ABS = 1.0      # <--- Dejamos pasar todo
# BG_MULT = 4
# MAX_BINS_FRAC = 0.25 # <--- Aquí dejamos todo, que decida el Teacher
# SMOOTH_MS = 0.2
# ADAPTIVE_WINDOW_MS = 15 

# Detección (Extracción en crudo, SIN UMBRAL DINÁMICO, para ver si el Oráculo sabe distinguir)
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=f"Target Prueba RFUAV", index=0,
)
# Magia Visual de tu Proyecto
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
fig_2d.show()

# Inferencia con modelo alumn_v1

In [ ]:
# IMPORTANTE: Cargamos el modelo TEACHER (Oráculo entrenado en SNR >= 0)
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v1\checkpoints\alumn_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = BurstCVCNN().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Rescatamos las estadísticas físicas (mean/std) originales con las que se entrenó el Teacher
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo TEACHER cargado con éxito en {device.type.upper()}")
print(f"   Época Óptima del guardado: {ckpt.get('epoch', 'N/A')}")
print(f"   Validation F1: {ckpt.get('val_f1', 0.0):.4f}")

In [ ]:
print("==================================================")
print("     VEREDICTO ALUMNO (Inferencia en vivo)        ")
print("==================================================")
drones_encontrados = 0
ruidos_encontrados = 0

if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La sala se considera VACÍA (RUIDO).")
else:
    # --- AJUSTE DE CLIPPING (Para ser idéntico al entrenamiento) ---
    global_nf      = float(np.clip(np.median(nf_v), 0, 15))
    global_ns_val  = float(np.clip(ns, 0, 5))
    global_H_mean  = float(np.clip(np.mean(H_smooth), 0, 15))
    global_p75_act = float(np.clip(np.percentile(n_active, 75), 0, 2048))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR Y NORMALIZAR ONDA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin: continue
            
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            
            # PADDING ESTÁNDAR (131072 muestras = 9.4 ms)
            TARGET_LEN = 131072
            C, L = pulso_normalizado.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso_normalizado.device)
                pulso_padded = torch.cat([pulso_normalizado, pad], dim=1)
            else:
                pulso_padded = pulso_normalizado[:, :TARGET_LEN]
                
            input_ia = pulso_padded.unsqueeze(0).to(device) 
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones en orden exacto)
            feat_array = np.array([
                np.clip(b['dur_ms'], 0, 75),
                np.clip(abs(b['z_peak']), 0, 30),
                np.clip(b['drop_b'], 0, 10),
                np.clip(b['n_act'], 0, 2048),
                global_nf, 
                global_ns_val, 
                global_H_mean, 
                global_p75_act
            ], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            # Normalización manual con stats del checkpoint
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. CLASIFICACIÓN
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")

print("-" * 50)
print(f"RESUMEN: {drones_encontrados} Drones | {ruidos_encontrados} Ruidos")

# Inferencia con modelo xin_v1

In [ ]:
from NoisyUAV.modelo_xin_v1.xin_cvcnn import XinCVCNN
from NoisyUAV.modelo_xin_v1.xin_dataset import (
    iq_to_xin_tensor, NFFT, HOP_LENGTH, SPEC_H, SPEC_W, DB_CLIP
)

In [ ]:
TARGET_NAMES = {0: "Target 0", 1: "Target 1", 2: "Target 2",
                3: "Target 3", 4: "Ruido (T4)", 5: "Target 5", 6: "Target 6"}
FS_HZ    = 14_000_000   # Frecuencia de muestreo [Hz]
NFFT     = 1024
HOP      = 512

In [ ]:
# IMPORTANTE: Cargamos el modelo TEACHER (Oráculo entrenado en SNR >= 0)
ruta_pesos = r"C:\repos\DroneDetectionRF\NoisyUAV\modelo_xin_v1\resultados\checkpoints\xin_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = XinCVCNN(num_classes=1, kernel_size=5).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Rescatamos las estadísticas físicas (mean/std) originales con las que se entrenó el Teacher
print(f"Modelo cargado — Epoch {ckpt['epoch']} | Best Val F1: {ckpt['best_val_f1']:.4f}")

In [ ]:
iq  = iq_tensor.clone()   # [2, N_SAMPLES]
N   = iq.shape[1]
t_axis_ms = np.linspace(0, N / FS_HZ * 1000, N)   # eje temporal en ms

print(f"\n  Muestras IQ : {N:,}  (~{N/FS_HZ*1000:.1f} ms)")
print(f"  RMS I       : {iq[0].pow(2).mean().sqrt():.4f}")
print(f"  RMS Q       : {iq[1].pow(2).mean().sqrt():.4f}")

In [ ]:
# ── Paso 1: Normalizacion RMS ────────────────────────────────────────────────
rms  = iq.pow(2).mean().clamp(min=1e-12).sqrt()
iq_n = iq / rms

# ── Paso 2: STFT compleja ────────────────────────────────────────────────────
sig_complex = torch.complex(iq_n[0], iq_n[1])
window      = torch.hann_window(NFFT)
stft        = torch.stft(
    sig_complex, n_fft=NFFT, hop_length=HOP, win_length=NFFT,
    window=window, center=False, return_complex=True, onesided=False
)   # [F, T]

# ── Paso 3-5: log-PSD normalizado ───────────────────────────────────────────
psd_lin  = stft.abs().pow(2)
psd_db   = 10.0 * torch.log10(psd_lin + 1e-12)
psd_max  = psd_db.max()
psd_clip = psd_db.clamp(min=psd_max - DB_CLIP)
psd_min  = psd_clip.min()
psd_norm = (psd_clip - psd_min) / (psd_max - psd_min).clamp(min=1e-8)   # [0,1]

# ── Paso 6: Resize bilineal a [SPEC_H, SPEC_W] ──────────────────────────────
psd_4d   = psd_norm.unsqueeze(0).unsqueeze(0)
psd_res  = F.interpolate(psd_4d, size=(SPEC_H, SPEC_W),
                         mode="bilinear", align_corners=False).squeeze()

# ── Paso 7-8: Sobel ──────────────────────────────────────────────────────────
Kx = torch.tensor([[1.,0.,-1.],[2.,0.,-2.],[1.,0.,-1.]]).view(1,1,3,3)
Ky = torch.tensor([[1.,2.,1.],[0.,0.,0.],[-1.,-2.,-1.]]).view(1,1,3,3)
img4 = psd_res.unsqueeze(0).unsqueeze(0)
pad  = F.pad(img4, (1,1,1,1), mode="reflect")
Gx   = F.conv2d(pad, Kx).squeeze()
Gy   = F.conv2d(pad, Ky).squeeze()
sobel_mag  = torch.sqrt(Gx**2 + Gy**2 + 1e-8)
sobel_norm = (sobel_mag - sobel_mag.min()) / (sobel_mag.max() - sobel_mag.min() + 1e-8)

# ── Ejes en unidades reales ──────────────────────────────────────────────────
T_frames    = stft.shape[1]
freq_bins   = np.fft.fftshift(np.fft.fftfreq(NFFT, d=1/FS_HZ)) / 1e6  # MHz
time_frames = np.linspace(0, N / FS_HZ * 1000, T_frames)               # ms

print(f"STFT shape: {stft.shape}  (F={NFFT} bins, T={T_frames} frames)")
print(f"Rango temporal: {time_frames[0]:.1f} – {time_frames[-1]:.1f} ms")
print(f"Rango frecuencial: {freq_bins[0]:.1f} – {freq_bins[-1]:.1f} MHz")
print(f"Tensor modelo [2,{SPEC_H},{SPEC_W}]  listo.")

In [ ]:
psd_plot = np.fft.fftshift(psd_norm.numpy(), axes=0)  # centrar frecuencia en 0

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    psd_plot,
    aspect="auto",
    origin="lower",
    cmap="inferno",
    extent=[time_frames[0], time_frames[-1],
            freq_bins[0],   freq_bins[-1]],
)
plt.colorbar(im, ax=ax, label="Potencia normalizada [0, 1]")
ax.set_title(
    f"Cajitas FHSS: bloques de energia en el espectrograma",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("Tiempo (ms)", fontsize=11)
ax.set_ylabel("Frecuencia (MHz)", fontsize=11)
ax.axhline(0, color="white", linewidth=0.5, alpha=0.4, linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
psd_res_plot   = psd_res.numpy()
sobel_res_plot = sobel_norm.numpy()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(
    f"Pipeline de transformacion | SNR={SNR_A_PROBAR} dB",
    fontsize=13, fontweight="bold"
)

# Panel 1: log-PSD redimensionado (entrada Canal 0 del modelo)
axes[0].imshow(psd_res_plot, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=1)
axes[0].set_title("Canal 0: log-PSD [0,1]\n(entrada real al modelo)", fontsize=11)
axes[0].set_xlabel("Frames temporales (256)")
axes[0].set_ylabel("Bins frecuenciales (256)")
plt.colorbar(
    plt.cm.ScalarMappable(cmap="inferno", norm=plt.Normalize(0, 1)),
    ax=axes[0], fraction=0.04
)

# Panel 2: Mapa de bordes Sobel (entrada Canal 1 del modelo)
axes[1].imshow(sobel_res_plot, aspect="auto", origin="lower", cmap="hot", vmin=0, vmax=1)
axes[1].set_title("Canal 1: Gradiente Sobel [0,1]\n(bordes de las cajitas)", fontsize=11)
axes[1].set_xlabel("Frames temporales (256)")
axes[1].set_ylabel("")
plt.colorbar(
    plt.cm.ScalarMappable(cmap="hot", norm=plt.Normalize(0, 1)),
    ax=axes[1], fraction=0.04
)

# Panel 3: Superposicion — PSD + bordes Sobel en rojo
axes[2].imshow(psd_res_plot, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=1)
# Umbral adaptativo: bordes fuertes (top 15%)
th_val    = float(np.percentile(sobel_res_plot, 85))
edge_mask = sobel_res_plot > th_val
overlay   = np.zeros((*psd_res_plot.shape, 4))
overlay[edge_mask] = [1.0, 0.0, 0.0, 0.75]   # rojo semitransparente
axes[2].imshow(overlay, aspect="auto", origin="lower")
axes[2].set_title("Superposicion: PSD + contornos Sobel\n(bordes del 15% mas alto en rojo)", fontsize=11)
axes[2].set_xlabel("Frames temporales (256)")
axes[2].set_ylabel("")
red_patch = mpatches.Patch(color="red", alpha=0.75, label=f"Bordes Sobel > {th_val:.2f}")
axes[2].legend(handles=[red_patch], fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
label_true = 1  # 1 si es Dron, 0 si es Ruido
label_name = {0: "Ruido", 1: "Drone"}
target_label_str = "RFUAV (Mavic 3 PRO)" # Nombre para el gráfico

# Convertimos iq_tensor [2, N] -> [2, 256, 256]
xin_tensor = iq_to_xin_tensor(iq_tensor, nfft=NFFT, hop_length=HOP_LENGTH,
                               spec_h=SPEC_H, spec_w=SPEC_W, db_clip=DB_CLIP)
batch = xin_tensor.unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    logit = model(batch)
    prob  = torch.sigmoid(logit).item()
    pred  = int(prob >= 0.5)
# Extraemos el espectrograma (canal I) para visualizar
psd_res_plot = xin_tensor[0].cpu().numpy()
# ── 3. IMPRESIÓN DE RESULTADOS ───────────────────────────────────────────────
print("=" * 50)
print(f"  Etiqueta real   : {label_name[label_true]} ({label_true})")
print(f"  Prediccion      : {label_name[pred]} ({pred})")
print(f"  P(drone)        : {prob:.4f}  ({prob*100:.1f}%)")
print(f"  Resultado       : {'CORRECTO' if pred == label_true else 'INCORRECTO'}")
print("=" * 50)
# ── 4. VISUALIZACIÓN COMPLETA ────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
fig.suptitle(
    f"Inferencia — {target_label_str} | SNR={SNR_A_PROBAR} dB | LIVE TEST",
    fontsize=13, fontweight="bold"
)
# Izquierda: Espectrograma
axes[0].imshow(psd_res_plot, aspect="auto", origin="lower", cmap="inferno")
axes[0].set_title("Espectrograma (entrada al modelo)", fontsize=11)
axes[0].set_xlabel("Frames temporales")
axes[0].set_ylabel("Bins frecuenciales")
# Borde de color según acierto
correct   = (pred == label_true)
box_color = "#2D6A4F" if correct else "#D62828"
for spine in axes[0].spines.values():
    spine.set_edgecolor(box_color); spine.set_linewidth(4)
# Texto de resultado sobre el plot
result_text = f"P(Drone) = {prob:.3f}\nPrediccion: {label_name[pred]}\nReal: {label_name[label_true]}"
axes[0].text(
    0.02, 0.97, result_text, transform=axes[0].transAxes,
    fontsize=10, verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, 
              edgecolor=box_color, linewidth=2),
    color="#1A1A1A"
)
# Derecha: Barra de probabilidad
bar_colors = ["#D62828", "#2D6A4F"]
bars = axes[1].barh(
    ["Ruido", "Drone"], [1 - prob, prob],
    color=[bar_colors[0], bar_colors[1]],
    edgecolor="black", linewidth=0.8, height=0.5
)
axes[1].set_xlim(0, 1)
axes[1].axvline(0.5, color="black", linestyle="--", linewidth=1.2, alpha=0.6)
axes[1].set_xlabel("Probabilidad", fontsize=11)
axes[1].set_title("Salida del modelo", fontsize=11)
# Etiquetas de valor en las barras
for bar, val in zip(bars, [1-prob, prob]):
    axes[1].text(
        min(val + 0.02, 0.95), bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center", fontsize=10, fontweight="bold"
    )
# Flecha indicadora de predicción
pred_y = ["Ruido", "Drone"][pred]
plt.tight_layout()
plt.show()